### Correlation Matrices

In [33]:
from pathlib import Path
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

path = Path('/content/drive/MyDrive/hydromind')
fig_path = path / "figures"
fig_path.mkdir(parents=True, exist_ok=True)

def correlation_analysis(household_df: pl.DataFrame, water_usage_matrix: pl.DataFrame, name: str) -> pd.core.frame.DataFrame:
    # compute pearson and spearman correlation matrices
    water_usage_np = water_usage_matrix.to_numpy()

    features = household_df.with_columns([
        pl.Series("mean_daily_water_usage", water_usage_np.mean(axis=1)),
        pl.Series("weekend_mean", water_usage_np[:, [i for i in range(365) if i % 7 in (5, 6)]].mean(axis=1)),
        pl.Series("weekday_mean", water_usage_np[:, [i for i in range(365) if i % 7 not in (5, 6)]].mean(axis=1))
    ]).select([
        "occupancy_count",
        "appliance_efficiency_score",
        "mean_daily_water_usage",
        "weekend_mean",
        "weekday_mean",
    ])

    numeric_df = features.to_pandas()
    # encode landscape type
    landscape_encoded = household_df.to_dummies("landscape_type", drop_first=True).to_pandas()
    combined_df = pd.concat([numeric_df, landscape_encoded], axis=1)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    sns.heatmap(combined_df.corr
        (method="pearson", numeric_only=True), 
        annot=True, 
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        ax=axes[0]
    )
    axes[0].set_title("Pearson Correlation")

    sns.heatmap(combined_df.corr
        (method="spearman", numeric_only=True),
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        ax=axes[1]
    )
    axes[1].set_title("Spearman Correlation")

    plt.tight_layout()
    plt.savefig(f"{fig_path}/Correlation Matrix ({name}).png", dpi=150)
    plt.close()

    return numeric_df.corr(method="pearson")

### Household Correlation Analysis

In [ ]:
%store -r df_north
%store -r df_south

north_household_correlation = correlation_analysis(household_df=df_north["household"], water_usage_matrix=df_north["water_usage"], name="North Hemisphere")
south_household_correlation = correlation_analysis(household_df=df_south["household"], water_usage_matrix=df_south["water_usage"], name="South Hemisphere")

print(north_household_correlation)
print(south_household_correlation)

### Conditional Distributions

In [35]:
def conditional_by_landscape(household_df: pl.DataFrame, water_usage_matrix: pl.DataFrame, name: str) -> None:
    # box plots of mean daily water usage by landscape type
    water_usage_np = water_usage_matrix.to_numpy()
    df = household_df.with_columns(
        pl.Series("mean_daily_water_usage", water_usage_np.mean(axis=1))
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    pd_df = df.select(["landscape_type", "mean_daily_water_usage"]).to_pandas()
    sns.boxplot(data=pd_df, x="landscape_type", y="mean_daily_water_usage", ax=ax, palette="viridis")
    ax.set_title("Mean Daily Usage by Landscape Type")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(f"{fig_path}/Conditional Distributions ({name}).png", dpi=150)
    plt.close()

In [ ]:
north_conditional_distribution = conditional_by_landscape(df_north["household"], df_north["water_usage"], name="North Hemisphere")
south_conditional_distribution = conditional_by_landscape(df_south["household"], df_south["water_usage"], name="South Hemisphere")

### Temporal Patterns

In [37]:
def temporal_patterns(env_df: pl.DataFrame, water_usage_matrix, hemisphere: str) -> None:
    # time-series plots for temperature, rainfall, water daily usage
    water_usage_np = water_usage_matrix.to_numpy()
    daily_mean_water_usage = water_usage_np.mean(axis=0)

    fig, axes = plt.subplots(3, 1, figsize=(14, 20), sharex=True)
    days = np.arange(365)

    axes[0].plot(days, env_df["daily_max_temp_celsius"].to_numpy(), color="orangered", lw=0.8)
    axes[0].set_ylabel("Temp (°C)")
    axes[0].set_title(f"{hemisphere.title()} - Daily Temperature")

    axes[1].bar(days, env_df["daily_rainfall_mm"].to_numpy(), color="steelblue", width=1.0, alpha=0.7)
    axes[1].set_ylabel("Rainfall (mm)")
    axes[1].set_title(f"{hemisphere.title()} - Daily Rainfall")

    axes[2].plot(days, daily_mean_water_usage, color="teal", lw=0.8)
    axes[2].set_ylabel("Mean Water Usage (L)")
    axes[2].set_xlabel("Day of the Year")
    axes[2].set_title(f"{hemisphere.title()} - Mean Daily Water Usage (All Households)")

    plt.tight_layout()
    plt.savefig(f"{fig_path}/Temporal Patterns ({hemisphere}).png", dpi=150)
    plt.close()

In [38]:
temporal_patterns(df_north["environment"], df_north["water_usage"], hemisphere="North Hemisphere")
temporal_patterns(df_south["environment"], df_south["water_usage"], hemisphere="South Hemisphere")